<a href="https://colab.research.google.com/github/import-dhruv/AI-Research/blob/main/generated_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗂️ Instruction Dataset Generator
**Generates a general-purpose Alpaca-format dataset — no API key needed!**

Combines:
- 📚 **HuggingFace** pre-built datasets (fast, high quality)
- 🤖 **Local LLM via `llama-cpp-python`** (generates new examples)

---
### ⚡ Quick Start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells (`Runtime → Run all`)
3. Dataset downloads automatically at the end!


## ⚙️ Configuration
**Edit these before running:**

In [1]:
# ═══════════════════════════════════════════════
#   ✏️  EDIT THESE SETTINGS
# ═══════════════════════════════════════════════

TOTAL_EXAMPLES   = 1000          # Total examples (500–2000 recommended)
HF_FRACTION      = 0.6           # Fraction from HuggingFace (0.0–1.0)
OUTPUT_FILENAME  = 'dataset.json'

# Model options (downloaded automatically):
#   'Qwen/Qwen2-0.5B-Instruct-GGUF'  +  'qwen2-0_5b-instruct-q4_k_m.gguf'  (~400MB, fastest)
#   'Qwen/Qwen2-1.5B-Instruct-GGUF'  +  'qwen2-1_5b-instruct-q4_k_m.gguf'  (~1GB,  recommended ✅)
MODEL_REPO = 'Qwen/Qwen2-1.5B-Instruct-GGUF'
MODEL_FILE = 'qwen2-1_5b-instruct-q4_k_m.gguf'

print(f'✅ Config: {TOTAL_EXAMPLES} examples | {int(HF_FRACTION*100)}% HuggingFace + {int((1-HF_FRACTION)*100)}% Local LLM')


✅ Config: 1000 examples | 60% HuggingFace + 40% Local LLM


## 📦 Cell 1 — Install Dependencies

In [2]:
print('Installing packages...')
import subprocess, sys

!pip install -q datasets huggingface_hub tqdm sentence-transformers

# Try GPU build of llama-cpp-python first, fall back to CPU
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python',
     '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('GPU build unavailable, installing CPU build...')
    !pip install -q llama-cpp-python

print('\n✅ All packages installed!')


Installing packages...

✅ All packages installed!


## 📥 Cell 2 — Download Local Model

In [3]:
from huggingface_hub import hf_hub_download
import os

print(f'Downloading {MODEL_FILE} from {MODEL_REPO}...')
print('This may take a few minutes on first run...')

model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir='/content/models'
)
size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f'\n✅ Model ready: {model_path}  ({size_mb:.0f} MB)')


This may take a few minutes on first run...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


qwen2-1_5b-instruct-q4_k_m.gguf:   0%|          | 0.00/986M [00:00<?, ?B/s]


✅ Model ready: /content/models/qwen2-1_5b-instruct-q4_k_m.gguf  (940 MB)


## 🔧 Cell 3 — Load Model & Define Helpers

In [4]:
import json, random, re, time
from tqdm.notebook import tqdm
from llama_cpp import Llama

print('Loading model...')
llm = Llama(model_path=model_path, n_ctx=2048, n_gpu_layers=-1, verbose=False)
print('✅ Model loaded!')

TASK_CATEGORIES = [
    'text summarization', 'question answering', 'creative writing',
    'grammar correction', 'brainstorming ideas', 'step-by-step explanation',
    'sentiment analysis', 'email writing', 'Python code generation',
    'math word problems', 'rewriting/paraphrasing', 'translation',
    'information extraction', 'text classification', 'giving advice',
]

BATCH_PROMPT = """Generate {n} instruction-following examples for the task: "{category}".
Return ONLY a valid JSON array. No explanation, no markdown.
Each element: {{"instruction": "...", "input": "...", "output": "..."}}
Generate {n} examples now:"""

def parse_json_safe(text):
    text = re.sub(r'^```(?:json)?', '', text.strip()).strip()
    text = re.sub(r'```$', '', text).strip()
    m = re.search(r'\[.*\]', text, re.DOTALL)
    return json.loads(m.group(0)) if m else []

def generate_batch(category, n=4):
    prompt = BATCH_PROMPT.format(n=n, category=category)
    out = llm(prompt, max_tokens=1200, temperature=0.8, stop=['\n\n\n'])
    try:
        examples = parse_json_safe(out['choices'][0]['text'])
    except Exception:
        return []
    return [
        {'instruction': str(e.get('instruction','')).strip(),
         'input':       str(e.get('input','')).strip(),
         'output':      str(e.get('output','')).strip()}
        for e in examples
        if e.get('instruction') and e.get('output')
    ]

print('✅ Helpers ready!')


Loading model...


llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Model loaded!
✅ Helpers ready!


## 📚 Cell 4 — Load HuggingFace Datasets

In [5]:
from datasets import load_dataset

HF_SOURCES = [('yahma/alpaca-cleaned', 'train'), ('tatsu-lab/alpaca', 'train')]
hf_target  = int(TOTAL_EXAMPLES * HF_FRACTION)
per_source = max(1, hf_target // len(HF_SOURCES))
hf_examples = []

for ds_name, split in HF_SOURCES:
    try:
        print(f'Loading {ds_name}...')
        ds = load_dataset(ds_name, split=split)
        ds = ds.shuffle(seed=42).select(range(min(per_source * 2, len(ds))))
        for row in ds:
            inst = str(row.get('instruction', '')).strip()
            inp  = str(row.get('input', '')).strip()
            out  = str(row.get('output', '')).strip()
            if inst and out:
                hf_examples.append({'instruction': inst, 'input': inp, 'output': out})
        print(f'  ✅ {len(hf_examples)} examples loaded so far')
    except Exception as e:
        print(f'  ⚠️ Skipped {ds_name}: {e}')

random.shuffle(hf_examples)
hf_examples = hf_examples[:hf_target]
print(f'\n✅ HuggingFace: using {len(hf_examples)} examples')


Loading yahma/alpaca-cleaned...


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

  ✅ 600 examples loaded so far
Loading tatsu-lab/alpaca...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

  ✅ 1200 examples loaded so far

✅ HuggingFace: using 600 examples


## 🤖 Cell 5 — Generate Local LLM Examples

In [6]:
local_target = TOTAL_EXAMPLES - len(hf_examples)
local_examples = []
cat_index = 0
cats = TASK_CATEGORIES.copy()
random.shuffle(cats)

print(f'Generating {local_target} examples locally...')
pbar = tqdm(total=local_target, desc='🤖 Local generation')

while len(local_examples) < local_target:
    cat = cats[cat_index % len(cats)]
    cat_index += 1
    n = min(4, local_target - len(local_examples))
    try:
        batch = generate_batch(cat, n)
        local_examples.extend(batch)
        pbar.update(len(batch))
    except Exception:
        pass

pbar.close()
local_examples = local_examples[:local_target]
print(f'\n✅ Generated {len(local_examples)} local examples')


Generating 400 examples locally...


🤖 Local generation:   0%|          | 0/400 [00:00<?, ?it/s]


✅ Generated 400 local examples


## 🧹 Cell 6 — Merge, Deduplicate & Save

In [7]:
# Merge
combined = hf_examples + local_examples
print(f'Combined: {len(hf_examples)} HF + {len(local_examples)} local = {len(combined)} total')

# Exact dedup
seen, deduped = set(), []
for ex in combined:
    key = ex['instruction'].lower().strip()
    if key not in seen:
        seen.add(key)
        deduped.append(ex)
print(f'After exact dedup: {len(deduped)} (removed {len(combined)-len(deduped)})')

# Semantic dedup
try:
    from sentence_transformers import SentenceTransformer, util
    import torch
    print('Running semantic dedup...')
    st = SentenceTransformer('all-MiniLM-L6-v2')
    embs = st.encode([e['instruction'] for e in deduped], convert_to_tensor=True,
                     batch_size=128, show_progress_bar=True)
    keep, kept = [], []
    for ex, emb in zip(deduped, embs):
        if not kept or util.cos_sim(emb, torch.stack(kept))[0].max().item() < 0.85:
            keep.append(ex); kept.append(emb)
    print(f'After semantic dedup: {len(keep)} (removed {len(deduped)-len(keep)})')
    deduped = keep
except Exception as e:
    print(f'Semantic dedup skipped: {e}')

# Final shuffle + trim
random.shuffle(deduped)
final = deduped[:TOTAL_EXAMPLES]

# Save
output_path = f'/content/{OUTPUT_FILENAME}'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(final, f, indent=2, ensure_ascii=False)

with_input  = sum(1 for e in final if e['input'])
avg_out_len = sum(len(e['output']) for e in final) // len(final)
print(f"""
╔══════════════════════════════════╗
║     ✨ Dataset Ready!            ║
╠══════════════════════════════════╣
║  Total examples : {len(final)}
║  With input     : {with_input}
║  Without input  : {len(final)-with_input}
║  Avg output len : {avg_out_len} chars
║  Saved to       : {OUTPUT_FILENAME}
╚══════════════════════════════════╝
""")


Combined: 600 HF + 400 local = 1000 total
After exact dedup: 951 (removed 49)
Running semantic dedup...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

After semantic dedup: 894 (removed 57)

╔══════════════════════════════════╗
║     ✨ Dataset Ready!            ║
╠══════════════════════════════════╣
║  Total examples : 894
║  With input     : 500
║  Without input  : 394
║  Avg output len : 326 chars
║  Saved to       : dataset.json
╚══════════════════════════════════╝



## 💾 Cell 7 — Download Dataset

In [8]:
from google.colab import files
files.download(output_path)
print('✅ Download started!')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


## 👀 Cell 8 — Preview Examples (Optional)

In [9]:
print('=' * 60)
for i, ex in enumerate(random.sample(final, min(5, len(final))), 1):
    print(f'\n[Example {i}]')
    print(f'  INSTRUCTION : {ex["instruction"]}')
    if ex['input']:
        preview = ex['input'][:100] + '...' if len(ex['input']) > 100 else ex['input']
        print(f'  INPUT       : {preview}')
    out_preview = ex['output'][:200] + '...' if len(ex['output']) > 200 else ex['output']
    print(f'  OUTPUT      : {out_preview}')
    print('-' * 60)



[Example 1]
  INSTRUCTION : Given a piece of text, classify it as 'favorable' or 'unfavorable'.
  INPUT       : I'm very satisfied with my purchase.
  OUTPUT      : favorable
------------------------------------------------------------

[Example 2]
  INSTRUCTION : Create a list of five tools commonly used for data visualisation
  OUTPUT      : 1. Tableau - a powerful and interactive data visualization software.
2. Microsoft Excel - a versatile spreadsheet tool with basic data visualization capabilities.
3. QlikView - a business intelligence...
------------------------------------------------------------

[Example 3]
  INSTRUCTION : Translate the sentence 'The cake is made of flour, sugar, and eggs' into Chinese.
  INPUT       : The cake is made of flour, sugar, and eggs
  OUTPUT      : 蛋糕是由面粉、糖和鸡蛋组成的
------------------------------------------------------------

[Example 4]
  INSTRUCTION : Allocate your time effectively.
  INPUT       : 2. Do your homework.
  OUTPUT      : 3. Study the

## ☁️ Cell 9 — Save to Google Drive (Optional)

In [10]:
# Run this ONLY if you want to save to Google Drive
from google.colab import drive
import shutil
drive.mount('/content/drive')
drive_path = f'/content/drive/MyDrive/{OUTPUT_FILENAME}'
shutil.copy(output_path, drive_path)
print(f'✅ Saved to Google Drive: {drive_path}')


Mounted at /content/drive
✅ Saved to Google Drive: /content/drive/MyDrive/dataset.json
